In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. Load the dataset
# Make sure the filename matches exactly what you uploaded
filename = 'source_credibility_dataset_200.csv'
try:
    df = pd.read_csv(filename)
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print(f"Error: {filename} not found. Please upload the CSV file to Colab files.")

# 2. Select Features and Target
# We use numeric columns + language. We exclude identifiers like URL and source_name.
features = ['past_fake', 'past_real', 'domain_age_years', 'followers', 'language']
target = 'credibility_label'

X = df[features].copy()
y = df[target]

# 3. Preprocessing
# We need to convert 'language' (text) into numbers using LabelEncoder
lang_encoder = LabelEncoder()
X['language'] = lang_encoder.fit_transform(X['language'])

# 4. Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train the Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 6. Evaluate the Model
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel Training Complete.")
print(f"Accuracy on Test Set: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Dataset loaded successfully.

Model Training Complete.
Accuracy on Test Set: 95.00%

Classification Report:
               precision    recall  f1-score   support

        High       0.94      0.94      0.94        18
         Low       1.00      1.00      1.00         8
      Medium       0.93      0.93      0.93        14

    accuracy                           0.95        40
   macro avg       0.96      0.96      0.96        40
weighted avg       0.95      0.95      0.95        40



In [2]:
# Save the model and the encoder (needed to process new data later)
model_filename = 'credibility_rf_model.pkl'
encoder_filename = 'lang_encoder.pkl'

joblib.dump(rf_model, model_filename)
joblib.dump(lang_encoder, encoder_filename)

print(f"Model saved as '{model_filename}'")
print(f"Encoder saved as '{encoder_filename}'")

# Optional: Download the files to your local computer
try:
    from google.colab import files
    files.download(model_filename)
    files.download(encoder_filename)
    print("Download started...")
except ImportError:
    print("Not running in Colab or file download not supported here.")

Model saved as 'credibility_rf_model.pkl'
Encoder saved as 'lang_encoder.pkl'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started...


In [10]:
import pandas as pd
import joblib

# 1. Load the saved model and encoder
# (This ensures we are using the stored version, not the one in memory)
loaded_model = joblib.load('credibility_rf_model.pkl')
loaded_encoder = joblib.load('lang_encoder.pkl')

print("Model and Encoder loaded successfully.")

# 2. Define a helper function to predict credibility
def check_channel_credibility(source_name, past_fake, past_real, age, followers, language):
    # Create a DataFrame for the single input to match training format
    input_data = pd.DataFrame([{
        'past_fake': past_fake,
        'past_real': past_real,
        'domain_age_years': age,
        'followers': followers,
        'language': language
    }])

    # Encode the language using the loaded encoder
    try:
        input_data['language'] = loaded_encoder.transform(input_data['language'])
    except ValueError:
        print(f"Error: Language '{language}' not recognized by encoder.")
        return None

    # Predict probabilities
    probabilities = loaded_model.predict_proba(input_data)[0]
    # Get class labels in the order of probabilities
    class_labels = loaded_model.classes_

    # Create a dictionary of class labels and their probabilities
    prediction_probabilities = {label: prob for label, prob in zip(class_labels, probabilities)}

    # Return only the 'High' credibility probability
    return prediction_probabilities.get('High', 0.0)

# 3. Test with some Sample News Channels
# You can change these values to test different scenarios
test_channels = [
    # Case A: A channel with lots of fake news (Expected: Low probability for High)
    {'name': 'Gossip King', 'past_fake': 50, 'past_real': 5, 'age': 1, 'followers': 5000, 'lang': 'Sinhala'},

    # Case B: A reputable channel (Expected: High probability for High)
    {'name': 'Daily Truth', 'past_fake': 2, 'past_real': 1500, 'age': 15, 'followers': 2000000, 'lang': 'English'},

    # Case C: A new channel with mixed history (Expected: Medium or Low probability for High)
    {'name': 'News Today', 'past_fake': 5, 'past_real': 20, 'age': 3, 'followers': 15000, 'lang': 'Tamil'}
]

print("\n--- Testing New Channels ---")
for channel in test_channels:
    credibility_score = check_channel_credibility(
        channel['name'],
        channel['past_fake'],
        channel['past_real'],
        channel['age'],
        channel['followers'],
        channel['lang']
    )
    print(f"Channel: {channel['name']:<15} | Credibility Score: {credibility_score * 100:.2f}%")

Model and Encoder loaded successfully.

--- Testing New Channels ---
Channel: Gossip King     | Credibility Score: 0.00%
Channel: Daily Truth     | Credibility Score: 86.00%
Channel: News Today      | Credibility Score: 2.00%


In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import joblib

# 1. Load the model and encoder
try:
    loaded_model = joblib.load('credibility_rf_model.pkl')
    loaded_encoder = joblib.load('lang_encoder.pkl')
except FileNotFoundError:
    print("Error: Model files not found. Please run the training cell first.")

# 2. Create Input Widgets
style = {'description_width': 'initial'}

w_name = widgets.Text(
    value='',
    placeholder='e.g., Daily News',
    description='Channel Name:',
    style=style
)

w_fake = widgets.IntText(
    value=0,
    description='Past Fake News Count:',
    style=style
)

w_real = widgets.IntText(
    value=0,
    description='Past Real News Count:',
    style=style
)

w_age = widgets.IntText(
    value=0,
    description='Domain Age (Years):',
    style=style
)

w_followers = widgets.IntText(
    value=0,
    description='Followers:',
    style=style
)

w_lang = widgets.Dropdown(
    options=['Sinhala', 'Tamil', 'English'], # Ensure these match your training data
    value='Sinhala',
    description='Language:',
    style=style
)

btn_predict = widgets.Button(
    description='Predict Credibility',
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    icon='check'
)

output = widgets.Output()

# 3. Define Button Click Event
def on_predict_clicked(b):
    with output:
        clear_output()

        # Get data from widgets
        input_data = pd.DataFrame([{
            'past_fake': w_fake.value,
            'past_real': w_real.value,
            'domain_age_years': w_age.value,
            'followers': w_followers.value,
            'language': w_lang.value
        }])

        try:
            # Encode language
            input_data['language'] = loaded_encoder.transform(input_data['language'])

            # Predict probabilities
            probabilities = loaded_model.predict_proba(input_data)[0]
            class_labels = loaded_model.classes_
            prediction_probabilities = {label: prob for label, prob in zip(class_labels, probabilities)}

            # Display Result
            print(f"--- Result for '{w_name.value}' ---")

            # Get the probability for 'High' credibility
            high_credibility_score = prediction_probabilities.get('High', 0.0)
            print(f"Credibility Score: {high_credibility_score * 100:.2f}%")

            # Optionally, still show all probabilities for detailed view if desired
            # print("Predicted Credibility Probabilities:")
            # for label, prob in prediction_probabilities.items():
            #     print(f"  {label}: {prob * 100:.2f}%")

        except Exception as e:
            print(f"Error during prediction: {e}")

btn_predict.on_click(on_predict_clicked)

# 4. Display the UI
print("Enter News Channel Details below to test:")
ui = widgets.VBox([w_name, w_fake, w_real, w_age, w_followers, w_lang, btn_predict])
display(ui, output)

Enter News Channel Details below to test:


Output()